# 01: Preprocess EyePACS

Crops all 35,126 labelled EyePACS training images to the fundus, resizes them to 512 px and
writes them as JPEG into zip shards of 2,000 images, plus a validated manifest.

**Settings** (right-hand panel): Accelerator **None**, Internet **On**.
**Input:** *Add Input* → *Competitions* → **Diabetic Retinopathy Detection**.
**Run:** *Save Version* → *Save & Run All (Commit)*. Takes about 1.5 h; the browser can be closed.
If the session stops early, run it again: finished shards are kept and skipped.

**Output** (`/kaggle/working/eyepacs512/`):
- `eyepacs_000.zip` … `eyepacs_017.zip`: processed images (`{patient}_{eye}.jpg`)
- `eyepacs_NNN.zip.records.parquet`: per-image records (size, hashes, fundus circle, errors)
- `manifest_eyepacs.parquet`: labels + image columns, schema-validated
- `failures.csv`, `summary.json`, `identical_phash.csv`

> RetinaGuard is a research prototype, **not a medical device**.

In [ ]:
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"
OUT_DIR = "/kaggle/working/eyepacs512"

In [ ]:
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}" "pandera>=0.20"

In [ ]:
# --- 1. Locate the competition data and labels --------------------------------
import subprocess
import time
from pathlib import Path

from retinaguard import __version__
from retinaguard.data.sources import eyepacs

INPUT_ROOT = Path("/kaggle/input")
candidates = [
    p
    for pattern in ("*/trainLabels.csv*", "*/*/trainLabels.csv*", "*/*/*/trainLabels.csv*")
    for p in INPUT_ROOT.glob(pattern)
]
assert candidates, "trainLabels.csv not found: is the competition attached under Add Input?"
labels_path = candidates[0]

git_sha = subprocess.run(
    ["git", "ls-remote", REPO_URL, GIT_REF], capture_output=True, text=True
).stdout.split("\t")[0]
print(f"retinaguard {__version__} @ {git_sha[:10]}")

t0 = time.perf_counter()
images = eyepacs.KaggleImages(labels_path.parent)
labels = eyepacs.read_labels(labels_path)
missing = set(labels["image_id"].str.removeprefix("eyepacs:")) - images.names()
print(f"layout {images.layout}, {len(images.names()):,} images, {len(labels):,} labels")
print(f"indexed in {time.perf_counter() - t0:.0f}s")
assert not missing, f"{len(missing)} labelled images missing, e.g. {sorted(missing)[:5]}"

In [ ]:
# --- 2. Process all images, shard by shard (resumable) -------------------------
import pandas as pd

from retinaguard.data.processing import build_shard, shard_ids

out_dir = Path(OUT_DIR)
shards = shard_ids(labels["image_id"])
all_records = []
start = time.perf_counter()
for i, ids in enumerate(shards):
    t = time.perf_counter()
    records = build_shard(out_dir, f"eyepacs_{i:03d}.zip", ids, images.read)
    all_records.append(records)
    done = sum(len(r) for r in all_records)
    elapsed = time.perf_counter() - start
    eta_min = elapsed / done * (len(labels) - done) / 60
    print(
        f"shard {i + 1:>2}/{len(shards)}: {len(ids)} images, "
        f"{records['error'].notna().sum()} failed, {time.perf_counter() - t:5.0f}s, "
        f"ETA {eta_min:4.0f} min",
        flush=True,
    )
records = pd.concat(all_records, ignore_index=True)

In [ ]:
# --- 3. Manifest, failures and summary ---------------------------------------
import json

from retinaguard.data.invariants import duplicate_image_ids
from retinaguard.data.processing import attach_records

manifest = attach_records(labels, records)
assert not duplicate_image_ids(manifest)
manifest.to_parquet(out_dir / "manifest_eyepacs.parquet", index=False)

failures = records[records["error"].notna()]
failures.to_csv(out_dir / "failures.csv", index=False)

ok = records[records["error"].isna()]
radius_ratio = 2 * ok["fundus_radius"] / ok["width"]
summary = {
    "retinaguard_version": __version__,
    "git_sha": git_sha,
    "preprocess_version": str(records["preprocess_version"].iloc[0]),
    "labelled_images": len(labels),
    "processed": len(ok),
    "failed": len(failures),
    "failure_reasons": failures["error"].str.split(":").str[0].value_counts().to_dict(),
    "exact_duplicate_raw_files": int(records["sha256"].duplicated().sum()),
    "identical_phash_pairs": int(ok["phash"].duplicated().sum()),
    "shards": len(shards),
    "output_gb": round(sum(p.stat().st_size for p in out_dir.glob("*.zip")) / 1e9, 2),
    "radius_ratio_p01_p50_p99": [
        round(float(q), 3) for q in radius_ratio.quantile([0.01, 0.5, 0.99])
    ],
    "minutes": round((time.perf_counter() - start) / 60, 1),
}
(out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print(failures[["image_id", "error"]].head(20).to_string())

In [ ]:
# --- 4. Spot check: a few processed images straight from the shards ------------
import zipfile

import matplotlib.pyplot as plt

from retinaguard.data.preprocess import decode_rgb, local_mean_normalise
from retinaguard.data.processing import split_processed_path

sample = manifest.dropna(subset=["processed_path"]).sample(4, random_state=0)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col, (_, row) in enumerate(sample.iterrows()):
    shard_file, member = split_processed_path(row["processed_path"])
    with zipfile.ZipFile(out_dir / shard_file) as archive:
        img = decode_rgb(archive.read(member))
    for ax, shown in zip(axes[:, col], (img, local_mean_normalise(img)), strict=True):
        ax.imshow(shown)
        ax.set_title(f"{row['image_id']} grade {row['dr_grade']}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
fig.savefig(out_dir.parent / "spot_check.png", dpi=80)
plt.show()

In [ ]:
# --- 5. Identical perceptual hashes: real duplicates or look-alikes? ----------
groups = ok[ok["phash"].duplicated(keep=False)].groupby("phash")["image_id"].apply(sorted)
patients = groups.apply(lambda ids: {i.split(":")[1].split("_")[0] for i in ids})
print(f"{len(groups)} groups, sizes {groups.str.len().value_counts().sort_index().to_dict()}")
print(f"groups within a single patient: {int((patients.str.len() == 1).sum())}")
ok[ok["phash"].isin(groups.index)].sort_values("phash").to_csv(
    out_dir / "identical_phash.csv", index=False
)


def load(image_id):
    shard_file, member = split_processed_path(
        manifest.loc[manifest["image_id"] == image_id, "processed_path"].iloc[0]
    )
    with zipfile.ZipFile(out_dir / shard_file) as archive:
        return decode_rgb(archive.read(member))


shown = groups.sample(min(6, len(groups)), random_state=0) if len(groups) else groups
if len(shown):
    fig, axes = plt.subplots(len(shown), 2, figsize=(6, 3 * len(shown)), squeeze=False)
    for row, ids in zip(axes, shown, strict=True):
        for ax, image_id in zip(row, ids[:2], strict=True):
            ax.imshow(load(image_id))
            ax.set_title(image_id, fontsize=8)
            ax.axis("off")
    plt.tight_layout()
    fig.savefig(out_dir.parent / "identical_phash.png", dpi=80)
    plt.show()

**After the version finishes:** later notebooks read this output directly: *Add Input* →
*Notebooks* → *Your Work* → **01: Preprocess EyePACS**. Then send back the printed
`summary.json` and the first failures, if any.